# 07 · Construcción del dataset definitivo del recomendador

Este notebook integra:

- `neighborhood_real_estate_enriched.csv`
- `transit_routes_by_neighborhood.csv`

La unión se realiza mediante `zone_key`.

## Criterio metodológico

Los registros con estado `no_route` se conservan en el dataset definitivo. Sus tiempos de transporte permanecen como valores ausentes (`NaN`): **no se imputan ni se inventan tiempos**.

El notebook genera:

1. Un dataset completo con todas las zonas.
2. Una vista preparada para Pareto y KNN, limitada a las filas con información de ruta utilizable.
3. Un fichero específico con los casos `no_route`, para mantener su trazabilidad.
4. Un informe de valores ausentes y validaciones de calidad.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

EXPECTED_NO_ROUTE = 37
JOIN_KEY = "zone_key"


## 1. Localización de los ficheros

El código busca automáticamente los CSV en las carpetas más habituales del proyecto. Si los tienes en otra ubicación, modifica las rutas de la siguiente celda.


In [ ]:
PROJECT_ROOT = Path.cwd()

def find_existing_file(filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / filename,
        PROJECT_ROOT / "data" / filename,
        PROJECT_ROOT / "data" / "processed" / filename,
        PROJECT_ROOT / "data" / "final" / filename,
        PROJECT_ROOT.parent / "data" / filename,
        PROJECT_ROOT.parent / "data" / "processed" / filename,
        PROJECT_ROOT.parent / "data" / "final" / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    searched = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        f"No se ha encontrado '{filename}'. Rutas revisadas:\n{searched}"
    )


REAL_ESTATE_PATH = find_existing_file("neighborhood_real_estate_enriched.csv")
TRANSIT_PATH = find_existing_file("transit_routes_by_neighborhood.csv")

OUTPUT_DIR = PROJECT_ROOT / "data" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_PATH = OUTPUT_DIR / "neighborhood_recommender_dataset.csv"
MODEL_READY_PATH = OUTPUT_DIR / "neighborhood_recommender_model_ready.csv"
NO_ROUTE_PATH = OUTPUT_DIR / "neighborhoods_no_route.csv"
MISSING_REPORT_PATH = OUTPUT_DIR / "missing_values_report.csv"

print("Inmobiliario:", REAL_ESTATE_PATH)
print("Transporte:  ", TRANSIT_PATH)
print("Salidas:     ", OUTPUT_DIR.resolve())


## 2. Carga de los datasets

In [ ]:
real_estate = pd.read_csv(REAL_ESTATE_PATH)
transit = pd.read_csv(TRANSIT_PATH)

print("Dimensiones del dataset inmobiliario:", real_estate.shape)
print("Dimensiones del dataset de transporte:", transit.shape)

display(real_estate.head())
display(transit.head())


## 3. Validaciones previas

In [ ]:
def validate_join_key(df: pd.DataFrame, df_name: str, key: str = JOIN_KEY) -> None:
    if key not in df.columns:
        raise KeyError(
            f"El dataset '{df_name}' no contiene la columna obligatoria '{key}'. "
            f"Columnas disponibles: {df.columns.tolist()}"
        )

    if df[key].isna().any():
        missing = int(df[key].isna().sum())
        raise ValueError(
            f"El dataset '{df_name}' contiene {missing} valores ausentes en '{key}'."
        )

    if df[key].astype(str).str.strip().eq("").any():
        raise ValueError(
            f"El dataset '{df_name}' contiene valores vacíos en '{key}'."
        )


validate_join_key(real_estate, "real_estate")
validate_join_key(transit, "transit")

# Normalización mínima para evitar errores por espacios accidentales.
real_estate[JOIN_KEY] = real_estate[JOIN_KEY].astype(str).str.strip()
transit[JOIN_KEY] = transit[JOIN_KEY].astype(str).str.strip()

print("Claves únicas en inmobiliario:", real_estate[JOIN_KEY].nunique())
print("Claves únicas en transporte:  ", transit[JOIN_KEY].nunique())

duplicates_real_estate = real_estate[real_estate.duplicated(JOIN_KEY, keep=False)].sort_values(JOIN_KEY)
duplicates_transit = transit[transit.duplicated(JOIN_KEY, keep=False)].sort_values(JOIN_KEY)

print("Filas duplicadas por zone_key en inmobiliario:", len(duplicates_real_estate))
print("Filas duplicadas por zone_key en transporte:  ", len(duplicates_transit))

if not duplicates_real_estate.empty:
    display(duplicates_real_estate)

if not duplicates_transit.empty:
    display(duplicates_transit)
    raise ValueError(
        "El dataset de transporte tiene más de una fila por zone_key. "
        "Debe revisarse antes de realizar una unión many-to-one."
    )


## 4. Identificación del estado de las rutas

Se intenta localizar automáticamente la columna que contiene valores como `ok`, `route_found` o `no_route`.


In [ ]:
STATUS_COLUMN_CANDIDATES = [
    "route_status",
    "status",
    "transit_status",
    "routing_status",
]

status_column = next(
    (column for column in STATUS_COLUMN_CANDIDATES if column in transit.columns),
    None,
)

if status_column is None:
    object_columns = transit.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        values = (
            transit[column]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .unique()
            .tolist()
        )
        if "no_route" in values:
            status_column = column
            break

if status_column is None:
    raise KeyError(
        "No se ha podido identificar la columna de estado de ruta. "
        "Revisa las columnas del fichero de transporte y añade su nombre "
        "a STATUS_COLUMN_CANDIDATES."
    )

transit[status_column] = (
    transit[status_column]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Columna de estado detectada:", status_column)
display(transit[status_column].value_counts(dropna=False).to_frame("filas"))


In [ ]:
no_route_before_merge = transit[status_column].eq("no_route")
no_route_count_before_merge = int(no_route_before_merge.sum())

print("Casos no_route documentados:", no_route_count_before_merge)

assert no_route_count_before_merge == EXPECTED_NO_ROUTE, (
    f"Se esperaban {EXPECTED_NO_ROUTE} casos no_route, "
    f"pero se han encontrado {no_route_count_before_merge}. "
    "No continúes hasta comprobar por qué ha cambiado el recuento."
)

display(
    transit.loc[no_route_before_merge]
    .sort_values(JOIN_KEY)
    .head(10)
)


## 5. Comprobación de los campos temporales

Los tiempos de las filas `no_route` deben continuar ausentes. Esta celda detecta las columnas relacionadas con duración o tiempo para poder auditarlas.


In [ ]:
TIME_COLUMN_TOKENS = (
    "time",
    "duration",
    "minute",
    "minutes",
    "minutos",
    "tiempo",
)

time_columns = [
    column
    for column in transit.columns
    if column != JOIN_KEY
    and any(token in column.lower() for token in TIME_COLUMN_TOKENS)
]

print("Columnas temporales detectadas:", time_columns)

if not time_columns:
    print(
        "AVISO: no se han detectado columnas temporales automáticamente. "
        "Comprueba sus nombres antes de usar el dataset en Pareto o KNN."
    )
else:
    invalid_no_route_times = transit.loc[no_route_before_merge, time_columns].notna().any(axis=1)
    invalid_count = int(invalid_no_route_times.sum())

    print("Filas no_route con algún tiempo informado:", invalid_count)

    if invalid_count > 0:
        display(
            transit.loc[no_route_before_merge]
            .loc[invalid_no_route_times, [JOIN_KEY, status_column] + time_columns]
        )
        raise ValueError(
            "Hay filas no_route con tiempos informados. "
            "Deben revisarse: no se van a conservar tiempos artificiales."
        )


## 6. Unión mediante `zone_key`

Se utiliza una unión izquierda desde el dataset inmobiliario. Así se conservan todas las zonas disponibles en la base inmobiliaria aunque no exista una ruta calculable.


In [ ]:
dataset = real_estate.merge(
    transit,
    on=JOIN_KEY,
    how="left",
    validate="m:1",
    indicator=True,
    suffixes=("", "_transit"),
)

print("Dimensiones tras la unión:", dataset.shape)
display(dataset["_merge"].value_counts(dropna=False).to_frame("filas"))


In [ ]:
unmatched_real_estate = dataset.loc[dataset["_merge"].eq("left_only")].copy()

print("Filas inmobiliarias sin coincidencia en transporte:", len(unmatched_real_estate))

if not unmatched_real_estate.empty:
    display(unmatched_real_estate[[JOIN_KEY]].drop_duplicates().sort_values(JOIN_KEY))


## 7. Conservación explícita de los `no_route`

Se crean indicadores de trazabilidad:

- `route_available`: existe una ruta utilizable.
- `route_missing_reason`: explica si la fila es `no_route` o si no encontró correspondencia en el fichero de transporte.
- `usable_for_pareto_knn`: indica si la fila tiene información de ruta suficiente para entrar directamente en los modelos.

Los `no_route` siguen formando parte del dataset completo.


In [ ]:
dataset["route_available"] = dataset[status_column].notna() & ~dataset[status_column].eq("no_route")

dataset["route_missing_reason"] = pd.Series(pd.NA, index=dataset.index, dtype="string")
dataset.loc[dataset[status_column].eq("no_route"), "route_missing_reason"] = "no_route"
dataset.loc[dataset["_merge"].eq("left_only"), "route_missing_reason"] = "missing_transit_match"

if time_columns:
    dataset["usable_for_pareto_knn"] = (
        dataset["route_available"]
        & dataset[time_columns].notna().all(axis=1)
    )
else:
    dataset["usable_for_pareto_knn"] = dataset["route_available"]

no_route_after_merge = dataset[status_column].eq("no_route")
no_route_count_after_merge = int(no_route_after_merge.sum())

print("no_route conservados después de la unión:", no_route_count_after_merge)
print("Filas utilizables directamente para Pareto/KNN:", int(dataset["usable_for_pareto_knn"].sum()))
print("Filas conservadas pero no utilizables directamente:", int((~dataset["usable_for_pareto_knn"]).sum()))

assert no_route_count_after_merge == EXPECTED_NO_ROUTE, (
    f"Tras la unión deberían conservarse {EXPECTED_NO_ROUTE} casos no_route, "
    f"pero aparecen {no_route_count_after_merge}. "
    "Revisa la correspondencia de zone_key antes de guardar."
)


## 8. Informe de valores ausentes

In [ ]:
def missing_values_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "unique_non_null": df.nunique(dropna=True).values,
    })

    return report.sort_values(
        ["missing_count", "column"],
        ascending=[False, True],
    ).reset_index(drop=True)


missing_report = missing_values_report(dataset)
display(missing_report)


In [ ]:
if time_columns:
    route_missing_audit = (
        dataset.groupby(
            ["route_missing_reason", status_column],
            dropna=False,
        )[time_columns]
        .agg(lambda series: int(series.isna().sum()))
    )

    display(route_missing_audit)


## 9. Validaciones finales de calidad

No se eliminan filas ni se imputan tiempos. Las validaciones comprueban que:

- no se han perdido filas del dataset inmobiliario;
- siguen existiendo exactamente 37 casos `no_route`;
- los tiempos de dichos casos siguen ausentes;
- la clave de unión permanece informada.


In [ ]:
assert len(dataset) == len(real_estate), (
    "La unión ha cambiado el número de filas del dataset inmobiliario."
)

assert dataset[JOIN_KEY].notna().all(), (
    "Existen valores ausentes en zone_key después de la unión."
)

assert int(dataset[status_column].eq("no_route").sum()) == EXPECTED_NO_ROUTE, (
    "No se han conservado exactamente los 37 casos no_route."
)

if time_columns:
    assert dataset.loc[
        dataset[status_column].eq("no_route"),
        time_columns,
    ].isna().all().all(), (
        "Alguna fila no_route contiene un tiempo de transporte. "
        "No se permite imputar o inventar esos valores."
    )

print("Todas las validaciones finales se han superado.")


## 10. Preparación de las salidas

### Dataset completo

Contiene todas las filas, incluidos los 37 `no_route`.

### Vista model-ready

Contiene únicamente las filas marcadas como `usable_for_pareto_knn = True`. Esta separación evita que los algoritmos reciban valores temporales ausentes, pero no elimina los `no_route` del dataset canónico.

### Casos no_route

Fichero independiente para documentación, revisión y trazabilidad.


In [ ]:
dataset_complete = (
    dataset
    .drop(columns="_merge")
    .sort_values(JOIN_KEY)
    .reset_index(drop=True)
)

dataset_model_ready = (
    dataset_complete
    .loc[dataset_complete["usable_for_pareto_knn"]]
    .copy()
    .reset_index(drop=True)
)

dataset_no_route = (
    dataset_complete
    .loc[dataset_complete[status_column].eq("no_route")]
    .copy()
    .reset_index(drop=True)
)

print("Dataset completo:      ", dataset_complete.shape)
print("Vista Pareto/KNN:      ", dataset_model_ready.shape)
print("Casos no_route:        ", dataset_no_route.shape)

display(dataset_complete.head())


In [ ]:
dataset_complete.to_csv(
    FINAL_DATASET_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_model_ready.to_csv(
    MODEL_READY_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_no_route.to_csv(
    NO_ROUTE_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_report.to_csv(
    MISSING_REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Ficheros guardados correctamente:")
print(" -", FINAL_DATASET_PATH.resolve())
print(" -", MODEL_READY_PATH.resolve())
print(" -", NO_ROUTE_PATH.resolve())
print(" -", MISSING_REPORT_PATH.resolve())


## 11. Resumen de control

Ejecuta esta celda al terminar y conserva su salida como evidencia del procesamiento.


In [ ]:
summary = pd.Series({
    "real_estate_rows": len(real_estate),
    "transit_rows": len(transit),
    "final_dataset_rows": len(dataset_complete),
    "unique_zone_keys": dataset_complete[JOIN_KEY].nunique(),
    "documented_no_route": int(dataset_complete[status_column].eq("no_route").sum()),
    "usable_for_pareto_knn": int(dataset_complete["usable_for_pareto_knn"].sum()),
    "not_usable_for_pareto_knn": int((~dataset_complete["usable_for_pareto_knn"]).sum()),
    "unmatched_transit_keys": int(dataset_complete["route_missing_reason"].eq("missing_transit_match").sum()),
})

display(summary.to_frame("value"))

assert summary["documented_no_route"] == EXPECTED_NO_ROUTE

print(
    "\nDataset definitivo construido sin imputar tiempos "
    "en los 37 casos no_route."
)
